In [ ]:
from pathlib import Path
import csv
import time
import torch
from tqdm import tqdm
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# ---- USER CONFIG ----
MODEL_ID = "AdaptLLM/biomed-Qwen2-VL-2B-Instruct"

MAX_NEW_TOKENS = 256
LOAD_IN_FP16 = True  # set False if running CPU-only
# ---------------------


In [ ]:
print("[INFO] Loading model...")

dtype = "auto"  # Let HF choose best dtype
if LOAD_IN_FP16 and torch.cuda.is_available():
    dtype = "auto"

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto",
    trust_remote_code=True,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model.eval()

print("[INFO] Model + processor loaded.")


In [ ]:
def run_inference_on_image(image_path, prompt, max_new_tokens=MAX_NEW_TOKENS):
    """
    Runs biomed-Qwen2-VL inference on a single image + prompt.
    Follows model-card example structure.
    """
    from PIL import Image

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": str(image_path)},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    # Build text input
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    # Process vision inputs
    image_inputs, video_inputs = process_vision_info(messages)

    # Prepare model inputs
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    # Move to same device as model
    device = next(model.parameters()).device
    inputs = inputs.to(device)

    # Generate
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
        )

    # Remove input token prefix
    trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, output_ids)
    ]

    # Decode
    result = processor.batch_decode(
        trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return result


In [ ]:
IMAGES_FOLDER = Path("./data/LLaVA-Med/images/test")   # change to your folder
OUTPUT_CSV = Path("./data/LLaVA-Med/Qwen2Med_Localization_Anydesk_Test_09Dec2025_V1.csv")

PROMPT = ( 
    """You are a clinical image assistant.

Look at the image and output ONLY the anatomical location.
- If an ADE is present: output only the location of the ADE.
- If no ADE: output only the body part shown.
- Output must be a short phrase (1–3 words), no explanation strictly.

Examples:
Image: (oral ulcers)
Response: The anatomical location of the ADE is in the tongue

Image: (swollen leg from drug edema)
Response: The anatomical location of the ADE is in the left leg

Image: (abdominal rash from ADE)
Response: The anatomical location of the ADE is in the stomach

Image: (no ADE, only arm visible)
Response: The anatomical location of the ADE is in the right arm

Image: (swollen leg from drug edema)
Response: The anatomical location of the ADE is in the right leg

Image: (no ADE, face visible)
Response: The anatomical location of the ADE is in the face """ )
# Validate folder

if not IMAGES_FOLDER.exists():
    raise FileNotFoundError(f"Image folder not found: {IMAGES_FOLDER}")

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}
image_list = sorted([p for p in IMAGES_FOLDER.iterdir() if p.suffix.lower() in image_exts])

print(f"[INFO] Found {len(image_list)} images.")

# Create CSV
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["imageName", "generatedText", "Prompt"])
    writer.writerow(["", "", PROMPT])   # prompt once at top

    # Process each image
    #i = 0
    #top = 10
    for img_path in tqdm(image_list, desc="Running inference"):
        #i = i + 1
        #if (i > top):
           # break
        try:
            gen = run_inference_on_image(img_path, PROMPT)
            gen = gen.replace("The anatomical location of the ADE is in the ", "").strip().rstrip(".")
            print(gen)
        except Exception as e:
            gen = f"[ERROR] {type(e).__name__}: {e}"

        writer.writerow([img_path.name, gen, ""])
        time.sleep(0.05)

print(f"[DONE] CSV saved to {OUTPUT_CSV.resolve()}")


In [ ]:
text = "The anatomical location of the ADE is in the left leg."

clean_text = text.replace("The anatomical location of the ADE is in the ", "").strip().rstrip(".")
print(clean_text)
